Bulk RNA-seq data processing for B.p and 5-AVAB mice

The raw sequencing reads were subjected to quality control and adaptor trimming using Trim Galore software (version 0.6.7) (https://github.com/FelixKrueger/TrimGalore). Following this, the cleaned reads were aligned to the Mus musculus reference genome (mm10) utilizing the STAR aligner (version 2.7.1a) with strand-specific parameters to ensure accurate mapping. The expression levels of the aligned reads were subsequently quantified using StringTie (version 2.1.7) to calculate Fragments Per Kilobase per Million (FPKM) values. Differential expression analysis was conducted using DESeq2 (version 1.42.1)105 to identify differentially expressed genes (DEGs) between B. pseudocatenulatum and vehicle samples, employing stringent statistical criteria with an adjusted P value < 0.05 and |log2(fold change)| > 0.5. The details of the DEGs are listed in Supplementary Table 8.


In [ ]:
#DESeq2 pipeline
rm(list=ls())
library(tidyverse)
library(ggplot2)
library(ggsci)
library(dplyr)
library(patchwork)
library(ggpubr)
library(DESeq2)
library(biomaRt) 
library(org.Hs.eg.db)

run_DESeq2<-function(input2,input3) {
  input2="Veh"
  input3="Bp"
  
  count_file_name="../BpCountMatrix.csv"
  raw_data <- read.csv(count_file_name,
                       row.names=1,check.names=F)
  metadata <- read.csv("../metadata.csv")
  
  length(colnames(raw_data))
  
  sampleNames <- colnames(raw_data)
  
  colData <- metadata%>%
    mutate(condition=ifelse(Group== input2,"Control","treated"))%>%
    dplyr::filter(Tissue == input3)%>%dplyr::select(x,condition)
  rownames(colData) <- colData$x
  colnames(colData)[1]="Samplename"
  colData
  
  raw_data=raw_data[,rownames(colData)]
  coldata <-colData
  
  raw_data<-as.matrix(raw_data)
  coldata$condition <- factor(coldata$condition,levels = c("Control","treated"))
  #condition????Ҫ??Ϊfactor???ܼ???
  all(colnames(raw_data) == rownames(coldata))
  print(all(colnames(raw_data) == rownames(coldata)))
  #check ?Ƿ???Ӧ
  colnames(raw_data)
  rownames(coldata)
  dds <- DESeqDataSetFromMatrix(countData = raw_data,
                                colData = coldata,
                                design= ~ condition)
  dds <- dds[rowSums(counts(dds)) > 1,]
  dds2 <- DESeq(dds)
  resultsNames(dds2) # lists the coefficients
  res <- results(dds2)
  rld <- rlog(dds2)
  p1<-plotPCA(rld,intgroup="condition")+theme_classic()
  p1
  
  ggsave("PCA_intestine.pdf",
         p1,
         width =6, height =8,dpi=300)
  
  # plotPCA(rld,intgroup="0")
  write.csv(assay(rld), paste0("Bp_vs_",input2,"_",input3,"_rld.csv"))
  
  # res <- res[order(res$padj),]
  res <- res[order(res$pvalue),]
  resdata <- merge(as.data.frame(res),as.data.frame(counts(dds2,normalize=TRUE)),by="row.names",sort=FALSE)
  names(resdata)[1]<-"GeneID"
  
  ########################## Exporting results to CSV files ##########################
  write.csv(resdata, paste0("Bp_vs_",input2,"_",input3,"_all_gene.csv",sep=""),row.names = F)
  
  diff_gene <-subset(resdata, padj < 0.05 & (log2FoldChange >= 0.5 | log2FoldChange <= -0.5))
  # diff_gene <-subset(resdata, pvalue < 0.05 & (log2FoldChange >= 0.5 | log2FoldChange <= -0.5))
  
  write.csv(diff_gene, paste0("Bp_vs_",input2,"_",input3,"_diff_gene.csv",sep=""),row.names = F)
  
  up_gene<-subset(diff_gene,diff_gene$log2FoldChange >= 0.5)
  write.csv(up_gene,paste0("Bp_vs_",input2,"_",input3,"_up_gene.csv",sep=""),row.names = F)
  
  down_gene<-subset(diff_gene,diff_gene$log2FoldChange <= -0.5)
  write.csv(down_gene,paste0("Bp_vs_",input2,"_",input3,"_down_gene.csv",sep=""),row.names = F)
  
  # ####################################################
  # #ע??
  # ####################################################
  
  listEnsemblArchives()
  listMarts(host = 'https://nov2020.archive.ensembl.org') #102
  
  listMarts(host = 'https://jan2020.archive.ensembl.org') #99
  
  mart<-useMart(host = 'https://nov2020.archive.ensembl.org',
                biomart="ENSEMBL_MART_ENSEMBL")
  dataset_list=as.data.frame(listDatasets(mart))
  dataset_list
  
  mart<-useMart(host = 'https://nov2020.archive.ensembl.org',
                biomart="ENSEMBL_MART_ENSEMBL",dataset="mmusculus_gene_ensembl")
  anns <- getBM(attributes = c("ensembl_gene_id","external_gene_name","description"),
                filters = "ensembl_gene_id", values=resdata$GeneID, mart=mart)
  
  anns <- anns[match(resdata$GeneID, anns[, 1]), ]
  colnames(anns) <- c("GeneID", "Gene Symbol", "Gene Description")
  
  all_gene_Description<-merge(anns,resdata,by="GeneID")
  write.csv(all_gene_Description, paste0("Bp_vs_",input2,"_",input3,"_all_gene_description.csv",sep=""),row.names = F)
  
  write.csv(colData,paste0("Bp_vs_",input2,"_",input3,"_colData.csv",sep=""),row.names = F)
  
  diff_gene_Description<-subset(all_gene_Description,padj < 0.05 & (log2FoldChange >= 0.5 | log2FoldChange <= -0.5))
  
  # diff_gene_Description<-subset(all_gene_Description,pvalue < 0.05 & (log2FoldChange >= 0.5 | log2FoldChange <= -0.5))
  
  write.csv(diff_gene_Description, paste0("Bp_vs_",input2,"_",input3,"_diff_gene_description.csv",sep=""),row.names = F)
  
  up_Description<-subset(diff_gene_Description,diff_gene_Description$log2FoldChange >=0.5)
  write.csv(up_Description,paste0("Bp_vs_",input2,"_",input3,"_up_gene_description.csv",sep=""),row.names = F)
  
  down_Description<-subset(diff_gene_Description,diff_gene_Description$log2FoldChange <=-0.5)
  write.csv(down_Description,paste0("Bp_vs_",input2,"_",input3,"_down_gene_description.csv",sep=""),row.names = F)}

for (i in c("WAT","Intestine","Brain","Muscle")){
  run_DESeq2("Veh",i)
} 

  
